# Reconstruction Example Using lftomo

Environment Requirements:

See x

In [1]:
from multiprocessing import shared_memory, Pool
from multiprocessing.managers import SharedMemoryManager
import os
from pathlib import Path
import sys

# Image Display
import cv2
from ipywidgets import Image
# Correct ordering if input not zero-padded
from natsort import natsorted
# Used for everything
import numpy as np
# Used to get processor data for setting multiprocessing.
import psutil
# Default file format is tif
import tifffile as tf
# Used for Recon
import tomopy

sys.path.insert(0, os.path.abspath('..'))

# Flexible logging to screen and file, in a parseable fixed-width format.
from lftomo.util.log import log
# Windowing to select only parts of an image at a time.
from lftomo.util.window import DataWindowSet, ProjOrder, ReconOrder, SinoOrder, get_tilt_pad, CTFov
# Handling flatfield backgrounds.
from lftomo.prep.flat import FLAT, weighted_normalize
# Shared Memory Handling
from lftomo.util.mem import SharedNP
# Reading and Writing files distributed
from lftomo.util.io import distribute_read, write_slice, np_convert, write_centers


## Setup

### Reconstruction Parameters

Starting with basics (input path, window size, output location)

In [2]:
log.set_logdir(Path("..", "data", "logs"))
log.header()
# smm = SharedMemoryManager()
# smm.start()

TYPE  |STEP                |   TIMESTAMP   |MEM USAGE|MEM FREE | STATEMENT 


In [3]:

window_size = psutil.cpu_count()
input_path = Path("..", "data", "input", "Sam19")
flat_path = Path("..", "data", "input", "Sam19_Flats")
output_base = Path("..", "data", "output")

# Field of View Angles.
fov = CTFov(Theta=None, Degrees=180, Count=1)

internal_dtype = np.float32		# Data type to use internally. 
cf_count = 8 					# Number of centerfind sections to generate.
proc_count = 12					# Number of simulataneous processes.
center_count = 20				# Number of centers to generate.

In [4]:
data_set = DataWindowSet.from_folder(input_path, ProjOrder, window_size, fov)
input_files = natsorted(Path(input_path).glob("**/*.tif*"))

### Flat Field Generation

In [5]:
flat_mem = SharedNP(f'flats_ipynb', internal_dtype, data_set.data_shape._replace(Theta=len(FLAT))).create()

with flat_mem as flat_set:
	for flat in list(FLAT):
		flat_set[flat.index, :, :] = tf.imread(flat_path.joinpath(f"{flat}_median.tiff")).astype(internal_dtype)
	
	# Precalculation of pregain and postgain with darks.
	flat_set[FLAT.PREGAIN.index] -= flat_set[FLAT.PREDARK.index]
	flat_set[FLAT.POSTGAIN.index] -= flat_set[FLAT.POSTDARK.index]


INFO  |Shared Memory       |00:00:00.106961|0002.00GB|0042.00GB|"Shared Memory Name flats_ipynb Pre-Existed"
INFO  |Shared Memory       |00:00:00.108550|0002.00GB|0042.00GB|"Shared Memory Name flats_ipynb Recreated"


In [6]:
with flat_mem as flat_set:
	flat_intermediate = output_base.joinpath("flat_intermediate")
	flat_intermediate.mkdir(parents=True, exist_ok=True)
	tf.imwrite(Path(flat_intermediate, "pregain.tiff"), flat_set[FLAT.PREGAIN.index])
	tf.imwrite(Path(flat_intermediate, "postgain.tiff"), flat_set[FLAT.POSTGAIN.index])
	tf.imwrite(Path(flat_intermediate, "predark.tiff"), flat_set[FLAT.PREDARK.index])
	tf.imwrite(Path(flat_intermediate, "postdark.tiff"), flat_set[FLAT.POSTDARK.index])

	if False:
		# Encode into in-memory PNG for Jupyter
		# _, pregain = cv2.imencode('.png', flat_set[FLAT.PREGAIN.index])
		# _, postgain = cv2.imencode('.png', flat_set[FLAT.POSTGAIN.index])

		# This 'works' but requires leveling
		Image(value=pregain, width=data_set.data_shape.X, height=data_set.data_shape.Y)

### Angle Calculation

In [7]:
# No angle dropping so skipping for now.

## Center Finding

Steps:

1. Read Slice(s)
2. Data Cleanup
3. Center Calculation
4. Angle Calculation
5. Reconstruct Slices
6. Normalize Slices


In [8]:
center_shape = SinoOrder(**data_set.data_shape._asdict())._replace(Y=cf_count)
cf_read_mem = SharedNP(f"cf_read_ipynb", data_set.dtype, center_shape).create()

cf_dataset = data_set.at_interval(cf_count)
centers = {}


for i, cf_data in enumerate(cf_dataset):
	log.write("Center Finding", f"{cf_data}, {cf_read_mem.buffer_address}")
	distribute_read(cf_read_mem[i], cf_data, input_files)
	with cf_read_mem as cf_slice:
		log.write("CF Test", f"{cf_slice.shape}")
		centers[cf_data.start] = tomopy.find_center_vo(cf_slice, i)

log.write("Center Finds", f"Vo CFs: {centers}")

INFO  |Shared Memory       |00:00:00.513329|0002.00GB|0042.00GB|"Shared Memory Name cf_read_ipynb Pre-Existed"
INFO  |Shared Memory       |00:00:00.514984|0002.00GB|0042.00GB|"Shared Memory Name cf_read_ipynb Recreated"
TIME  |Center Finding      |00:00:00.516864|0002.00GB|0042.00GB|"ProjOrder(Theta=1501, Y=1, X=2048) (uint16) at 0, slice(None, None, None)"
TIME  |CF Test             |00:00:00.658141|0002.00GB|0041.00GB|"(8, 1501, 2048)"
TIME  |Center Finding      |00:00:01.107984|0002.00GB|0041.00GB|"ProjOrder(Theta=1501, Y=1, X=2048) (uint16) at 256, slice(None, None, None)"
TIME  |CF Test             |00:00:01.232857|0002.00GB|0041.00GB|"(8, 1501, 2048)"
TIME  |Center Finding      |00:00:01.730949|0002.00GB|0041.00GB|"ProjOrder(Theta=1501, Y=1, X=2048) (uint16) at 512, slice(None, None, None)"
TIME  |CF Test             |00:00:01.904017|0002.00GB|0041.00GB|"(8, 1501, 2048)"
TIME  |Center Finding      |00:00:02.506801|0002.00GB|0042.00GB|"ProjOrder(Theta=1501, Y=1, X=2048) (uint16) a

In [9]:
cf_mem = SharedNP(f"cf_ipynp", internal_dtype, SinoOrder(**data_set.data_shape._asdict())._replace(Y=cf_count)).create()

with Pool(proc_count) as pool:
	pool.starmap(weighted_normalize, [(cf_mem, flat_mem, cf_read_mem, cf_dataset.full_window(), i, True) for i in range(cf_mem.shape.Theta)])
	log.write("Normalize", f"{cf_dataset.full_window().internal.Y()}|{cf_dataset.full_window().Y()}")


INFO  |Shared Memory       |00:00:06.076751|0002.00GB|0041.00GB|"Shared Memory Name cf_ipynp Pre-Existed"
INFO  |Shared Memory       |00:00:06.078432|0002.00GB|0041.00GB|"Shared Memory Name cf_ipynp Recreated"
TIME  |Normalize           |00:00:06.151130|0002.00GB|0041.00GB|"slice(0, 8, None)|slice(0, 8, None)"


In [10]:
base_center = tomopy.find_center_pc(tf.imread(input_files[0]), tf.imread(input_files[1]))
log.write("Center Find", f"Tomopy Center {base_center}")

TIME  |Center Find         |00:00:07.838165|0002.00GB|0041.00GB|"Tomopy Center 1023.5"


In [11]:
for cf_data in cf_dataset:
	log.write("Center Writing", f"{len(data_set.angles)}_{cf_mem.shape})")
	center_path = output_base.joinpath("cf_data", f"s{str(cf_data.start).zfill(5)}")
	center_path.mkdir(parents=True, exist_ok=True)
	write_centers(cf_mem[i], center_path, np.linspace(base_center - 5.0, base_center + 5.0, center_count), data_set.angles)

	center_path = output_base.joinpath("cf_data", f"noNorm_s{str(cf_data.start).zfill(5)}")
	center_path.mkdir(parents=True, exist_ok=True)
	write_centers(cf_read_mem[i], center_path, np.linspace(base_center - 5.0, base_center + 5.0, center_count), data_set.angles)

TIME  |Center Writing      |00:00:07.856507|0002.00GB|0041.00GB|"1501_SinoOrder(Y=8, Theta=1501, X=2048))"
INFO  |Shared Memory       |00:00:14.498128|0003.00GB|0041.00GB|"Unlinking cf_write_mem"
STATUS|Shared Memory       |00:00:14.499432|0003.00GB|0041.00GB|"Shared Memory cf_write_mem Is None"
INFO  |Shared Memory       |00:00:14.501740|0003.00GB|0041.00GB|"Shared Memory Name cf_write_mem Pre-Existed"
INFO  |Shared Memory       |00:00:14.521048|0004.00GB|0041.00GB|"Shared Memory Name cf_write_mem Recreated"
INFO  |Shared Memory       |00:00:21.006780|0003.00GB|0041.00GB|"Unlinking cf_write_mem"
STATUS|Shared Memory       |00:00:21.007886|0003.00GB|0041.00GB|"Shared Memory cf_write_mem Is None"
TIME  |Center Writing      |00:00:21.009417|0003.00GB|0041.00GB|"1501_SinoOrder(Y=8, Theta=1501, X=2048))"
INFO  |Shared Memory       |00:00:21.010826|0003.00GB|0041.00GB|"Shared Memory Name cf_write_mem Pre-Existed"
INFO  |Shared Memory       |00:00:21.030652|0004.00GB|0041.00GB|"Shared Memory

In [12]:
cf_mem.close()
cf_mem.unlink()
cf_read_mem.close()
cf_read_mem.unlink()

STATUS|Shared Memory       |00:01:49.442018|0003.00GB|0041.00GB|"Shared Memory cf_ipynp Is None"
STATUS|Shared Memory       |00:01:49.449420|0003.00GB|0041.00GB|"Shared Memory cf_read_ipynb Unlinked"


## Reconstruction

Loop, looking at small windows each time.

1. Read Data
2. Clean Data (Pre-Rotation Steps)
3. Rotate Data
4. Clean Data (Post-Rotation Steps)
5. Reconstruct Slices
6. Normalize Slices

In [13]:

if False:
	with (SharedNP(f"sino_ipynb", internal_dtype, SinoOrder(**data_set.window_shape._asdict())) as sino_mem,
		SharedNP(f"work_ipynb", data_set.dtype, data_set.window_shape) as workmem_read,
		SharedNP(f"recon_ipynb", internal_dtype, ReconOrder(Y=data_set.window_size, Z=data_set.window_shape.X, X=data_set.window_shape.X)) as recon_mem):

		for j, data in enumerate(data_set):
			log.write("Reading Data", f"{data_set.window_shape} : {data}")
			distribute_read(workmem_read[0:data_set.window_shape.Theta], data, input_files)

			with workmem_read as input_data, recon_mem as output_data:
				output_data[:, :, :] = tomopy.recon(input_data[:, :, :], data_set.angles, center=1032.5, algorithm="gridrec", ncore=proc_count)	

			with Pool(proc_count) as pool:
				output_base.joinpath("recon").mkdir(exist_ok=True, parents=True)
				log.write("Writing Recon", f"{data.Y()}")

				pool.starmap(write_slice, [(recon_mem[0:data_set.window_size], output_base.joinpath("recon", f"{str(i).zfill(5)}.tif"), i - data.Y().start)
												for i in data.Y()])


In [14]:
flat_mem.close()
flat_mem.unlink()

STATUS|Shared Memory       |00:01:49.493076|0003.00GB|0041.00GB|"Shared Memory flats_ipynb Unlinked"


In [15]:
sino_mem.close()
sino_mem.unlink()
workmem_read.close()
workmem_read.unlink()
recon_mem.close()
recon_mem.unlink()

NameError: name 'sino_mem' is not defined